In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

print("--- INITIATING: 02 ALGORITHMIC BENCHMARKING ---\n")

# ==========================================
# 1. DIRECT FILE UPLOAD
# ==========================================
try:
    from google.colab import files
    import io
    print("⏳ Step 1: Please upload your TRAINING Dataset CSV...")
    uploaded_train = files.upload()
    train_filename = list(uploaded_train.keys())[0]
    df_train = pd.read_csv(io.BytesIO(uploaded_train[train_filename]))

    print("⏳ Step 2: Please upload your TESTING Dataset CSV...")
    uploaded_test = files.upload()
    test_filename = list(uploaded_test.keys())[0]
    df_test = pd.read_csv(io.BytesIO(uploaded_test[test_filename]))
except ImportError:
    train_filename = input("Enter TRAINING CSV file: ")
    df_train = pd.read_csv(train_filename)
    test_filename = input("Enter TESTING CSV file: ")
    df_test = pd.read_csv(test_filename)

df_train.columns = df_train.columns.str.strip()
df_test.columns = df_test.columns.str.strip()

# ==========================================
# 2. PREPROCESSING & ALIGNMENT
# ==========================================
df_train_encoded = pd.get_dummies(df_train, columns=['Soil_Type'], dtype=int)
df_test_encoded = pd.get_dummies(df_test, columns=['Soil_Type'], dtype=int)
df_train_encoded, df_test_encoded = df_train_encoded.align(df_test_encoded, join='left', axis=1, fill_value=0)

feature_cols = ['Pile_Length_m', 'Pile_Size_mm', 'N60_Shaft', 'N60_Tip']
soil_cols = [col for col in df_train_encoded.columns if 'Soil_Type_' in col]
X_cols = feature_cols + soil_cols

X_train = df_train_encoded[X_cols]
y_train = df_train_encoded['Q_gross_kN']
X_test = df_test_encoded[X_cols]
y_test = df_test_encoded['Q_gross_kN']

# ==========================================
# 3. MODEL TOURNAMENT
# ==========================================
models = {
    "Decision Tree (DT)": DecisionTreeRegressor(random_state=42),
    "Random Forest (RF)": RandomForestRegressor(n_estimators=100, random_state=42),
    "Support Vector Machine (SVM)": SVR(kernel='rbf', C=1000),
    "K-Nearest Neighbors (KNN)": KNeighborsRegressor(n_neighbors=5),
    "XGBoost (Champion)": XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=5, random_state=42)
}

print("⚡ Training and evaluating multiple architectures...")
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model Architecture": name,
        "RMSE (kN)": round(rmse, 2),
        "R-Squared (R2)": round(r2, 4)
    })

# ==========================================
# 4. RESULTS OUTPUT
# ==========================================
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="RMSE (kN)", ascending=True).reset_index(drop=True)

print("\n--- 📊 BENCHMARKING RESULTS ---\n")
display(df_results)
print("\n✅ CONCLUSION: XGBoost selected as the Champion Model.")

--- INITIATING: 02 ALGORITHMIC BENCHMARKING ---

⏳ Step 1: Please upload your TRAINING Dataset CSV...


Saving Full detail-Full_Detail.csv to Full detail-Full_Detail.csv
⏳ Step 2: Please upload your TESTING Dataset CSV...


Saving Testing_18BH.csv to Testing_18BH.csv
⚡ Training and evaluating multiple architectures...

--- 📊 BENCHMARKING RESULTS ---



,Model Architecture,RMSE (kN),R-Squared (R2)
0,XGBoost (Champion),255.45,0.9884
1,Random Forest (RF),330.67,0.9806
2,Decision Tree (DT),425.36,0.9679
3,Support Vector Machine (SVM),653.73,0.9243
4,K-Nearest Neighbors (KNN),756.45,0.8986



✅ CONCLUSION: XGBoost selected as the Champion Model.
